# Let's reproduce GPT-2 (124M)（跟敲）

导读见 `guide.ipynb`。

这一章的参考实现是 `build-nanogpt/`（本地 clone，不进版本控制），
它的 **44 个 commit 就是视频的时间轴**——`git log --reverse --oneline` 一行一个知识点。


using device: cuda
loaded 338025 tokens
1 epoch = 2640 batches
step 0, loss: 10.960028648376465
step 1, loss: 9.687705993652344
step 2, loss: 9.082903861999512
step 3, loss: 9.145987510681152
step 4, loss: 8.626201629638672
step 5, loss: 8.331698417663574
step 6, loss: 8.897953033447266
step 7, loss: 8.837981224060059
step 8, loss: 8.116044998168945
step 9, loss: 8.042160987854004
step 10, loss: 8.380850791931152
step 11, loss: 7.435605049133301
step 12, loss: 7.824565887451172
step 13, loss: 7.458940505981445
step 14, loss: 7.531877040863037
step 15, loss: 7.366678714752197
step 16, loss: 7.43679666519165
step 17, loss: 8.293569564819336
step 18, loss: 7.202801704406738
step 19, loss: 7.887032508850098
step 20, loss: 7.505932807922363
step 21, loss: 7.822871685028076
step 22, loss: 6.425384998321533
step 23, loss: 6.8777995109558105
step 24, loss: 6.827327728271484
step 25, loss: 6.7018537521362305
step 26, loss: 6.814748287200928
step 27, loss: 7.621226787567139
step 28, loss: 7.1739

In [25]:
logits.dtype

torch.float32

In [ ]:
# prefix tokens
import tiktoken
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode("Hello, I'm a language model,")
tokens = torch.tensor(tokens, dtype=torch.long) # (8, )
tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1) # (5, 8)
x = tokens.to('cuda')

# generate! right now x is (B, T) where B = 5, T = 8
# set the seed to 42
torch.manual_seed(42)
while x.size(1) < max_length:
    # forward the model to get the logits
    with torch.no_grad():
        logits = model(x) # (B, T, vocab_size)
        # take the logits at the last position
        logits = logits[:, -1, :] # (B, vocab_size)
        # get the probabilities
        probs = F.softmax(logits, dim=-1)
        # do top-k sampling of 50 (huggingface pipeline default)
        # topk_probs here becomes (5, 50), topk_indices is (5, 50)
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
        # select a token from the top-k probabilities
        ix = torch.multinomial(topk_probs, 1) #(B, 1)
        # gather the corresponding indices
        xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
        # append to the sequence
        x = torch.cat((x, xcol), dim=1)

# print the generated text
for i in range(num_return_sequences):
    tokens = x[i, :max_length].tolist()
    decoded = enc.decode(tokens)
    print(">", decoded)